# Extract parish ministries

Predict ministries from the page information generated by `02_build_page_list.ipynb`. Inputs are read from `data/02_build_page_list/`; prediction outputs are saved in `data/03_extract_ministries/`. Run this notebook from the project root.


## 0 Environment setup
Install project dependencies with `%pip install -e .` if needed and load the project-root `.env` without overriding existing environment variables. This notebook uses local Ollama with `qwen3:8b`, following `10_ollama_test.ipynb`. Start Ollama and make sure the model is installed before running predictions.


In [1]:
from dotenv import load_dotenv

_ = load_dotenv(".env", override=False)


## 1 Predict ministries from parish page information
Use the local Ollama setup from `10_ollama_test.ipynb`: start Ollama with `qwen3:8b` installed, then install project dependencies with `%pip install -e .` if needed. Predictions use `chat`, a Pydantic JSON schema, validation, and temperature 0. See [Ollama structured outputs](https://ollama.com/blog/structured-outputs).

Set `PAGE_LIMIT` to a nonnegative integer to process only the first N discovered parish pages, or `None` for all pages. The limit applies across all parishes and is applied before LLM requests. Each selected page gets one request and one output row, even when no ministries are predicted or a request fails.

Load parish and discovery fields from `data/02_build_page_list/1_site_pages.csv` and join the available page fields from `data/02_build_page_list/2_page_content.json` by `page_url`. For now, predictions use URLs only; optional `final_url`, `page_title`, and `clean_text` fields are included automatically when present. This step does not download page content. URL-only predictions are provisional and are not evidence that a ministry exists.

Each predicted ministry has a standardized name and exactly one category from the fixed taxonomy in `utils/ministry_prediction.py`. Multiple ministries on a page are stored as a JSON array of `{ "name": "Altar Servers", "category": "Liturgy and Worship" }` objects in the CSV's `predicted_ministries` column. `[]` with status `ok` means no ministries were predicted; status `error` includes the failure in `prediction_error`.

Outputs in `data/03_extract_ministries/` (replaced on each run):
- `1_page_ministry_predictions.csv`: one row per selected parish page with parish info, page info, predictions, model, and status.
- `1_ministries_categories.csv`: all distinct predicted ministry/category pairs with the number of page rows reporting each pair.
- `1_ministry_categories.csv`: all categories actually predicted, with counts of distinct ministry names and page rows.


In [ ]:
import json
from pathlib import Path

import pandas as pd

from utils.ministry_prediction import predict_ministries

llm_model = "qwen3:8b"
PAGE_LIMIT = 5  # Set to None for all parish pages, or 0 to export empty results.

if PAGE_LIMIT is not None and (type(PAGE_LIMIT) is not int or PAGE_LIMIT < 0):
    raise ValueError("PAGE_LIMIT must be None or a nonnegative integer")

input_dir = Path("data/02_build_page_list")
output_dir = Path("data/03_extract_ministries")
site_pages_df = pd.read_csv(input_dir / "1_site_pages.csv", keep_default_na=False)
selected_pages = site_pages_df if PAGE_LIMIT is None else site_pages_df.head(PAGE_LIMIT)
with (input_dir / "2_page_content.json").open(encoding="utf-8") as file:
    page_content_records = json.load(file)
content_df = pd.DataFrame(page_content_records, columns=["page_url", "final_url", "page_title", "clean_text"])
content_df = content_df.fillna("").drop_duplicates()
# Identical URLs can occur for several parishes. Do not multiply page rows on the join.
selected_pages = selected_pages.merge(content_df, on="page_url", how="left", validate="many_to_one").fillna("")
prediction_rows = []
ministry_rows = []

for page_number, page in enumerate(selected_pages.to_dict("records"), start=1):
    ministries = []
    status = "ok"
    error = ""
    try:
        result = predict_ministries(page, model=llm_model)
        ministries = [ministry.model_dump() for ministry in result.ministries]
    except Exception as exc:
        # Preserve one output row per page, including connection and validation failures.
        status = "error"
        error = f"{type(exc).__name__}: {exc}"
    prediction_rows.append({
        **page,
        "predicted_ministries": json.dumps(ministries, ensure_ascii=False),
        "llm_model": llm_model,
        "prediction_status": status,
        "prediction_error": error,
    })
    ministry_rows.extend(ministries)
    print(f"{page_number}/{len(selected_pages)} {page['name']} {page['page_url']}: {len(ministries)} ministries ({status})")

prediction_columns = list(selected_pages.columns) + [
    "predicted_ministries", "llm_model", "prediction_status", "prediction_error",
]
page_predictions_df = pd.DataFrame(prediction_rows, columns=prediction_columns)
ministries_df = pd.DataFrame(ministry_rows, columns=["name", "category"])
ministries_categories_df = (
    ministries_df.groupby(["name", "category"]).size().reset_index(name="page_count")
    .sort_values(["category", "name"]).reset_index(drop=True)
)
ministry_categories_df = (
    ministries_df.groupby("category").agg(ministry_count=("name", "nunique"))
    .reset_index()
)
# A page can predict several ministries in the same category; count that page only once.
category_page_rows = [
    {"category": category}
    for row in prediction_rows
    for category in {m["category"] for m in json.loads(row["predicted_ministries"])}
]
category_page_counts = (
    pd.DataFrame(category_page_rows, columns=["category"])
    .groupby("category").size().reset_index(name="page_count")
)
ministry_categories_df = ministry_categories_df.merge(category_page_counts, on="category")
output_dir.mkdir(parents=True, exist_ok=True)
page_predictions_df.to_csv(output_dir / "1_page_ministry_predictions.csv", index=False)
ministries_categories_df.to_csv(output_dir / "1_ministries_categories.csv", index=False)
ministry_categories_df.to_csv(output_dir / "1_ministry_categories.csv", index=False)
print(f"Saved {len(page_predictions_df)} page predictions, {len(ministries_categories_df)} ministry/category pairs, and {len(ministry_categories_df)} categories to {output_dir}")
page_predictions_df.head()


1/20 Father Purcell Memorial Center for Exceptional Children https://fatherpurcell.org/: 1 ministries (ok)
2/20 Father Purcell Memorial Center for Exceptional Children https://fatherpurcell.org/admission: 0 ministries (ok)
3/20 Father Purcell Memorial Center for Exceptional Children https://fatherpurcell.org/blog: 0 ministries (ok)
4/20 Father Purcell Memorial Center for Exceptional Children https://fatherpurcell.org/bulletins: 0 ministries (ok)
5/20 Father Purcell Memorial Center for Exceptional Children https://fatherpurcell.org/calendar: 0 ministries (ok)
6/20 Father Purcell Memorial Center for Exceptional Children https://fatherpurcell.org/contact: 0 ministries (ok)
7/20 Father Purcell Memorial Center for Exceptional Children https://fatherpurcell.org/events: 0 ministries (ok)
8/20 Father Purcell Memorial Center for Exceptional Children https://fatherpurcell.org/history: 0 ministries (ok)
9/20 Father Purcell Memorial Center for Exceptional Children https://fatherpurcell.org/news: 0

,parish_index,name,diocese,site_url,page_url,sources,final_url,page_title,clean_text,predicted_ministries,llm_model,prediction_status,prediction_error
0,0,Father Purcell Memorial Center for Exceptional...,Archdiocese of Mobile,https://fatherpurcell.org/,https://fatherpurcell.org/,"homepage, homepage_link, sitemap",,,,"[{""name"": ""Exceptional Children Support"", ""cat...",qwen3:8b,ok,
1,0,Father Purcell Memorial Center for Exceptional...,Archdiocese of Mobile,https://fatherpurcell.org/,https://fatherpurcell.org/admission,"homepage_link, sitemap",,,,[],qwen3:8b,ok,
2,0,Father Purcell Memorial Center for Exceptional...,Archdiocese of Mobile,https://fatherpurcell.org/,https://fatherpurcell.org/blog,sitemap,,,,[],qwen3:8b,ok,
3,0,Father Purcell Memorial Center for Exceptional...,Archdiocese of Mobile,https://fatherpurcell.org/,https://fatherpurcell.org/bulletins,sitemap,,,,[],qwen3:8b,ok,
4,0,Father Purcell Memorial Center for Exceptional...,Archdiocese of Mobile,https://fatherpurcell.org/,https://fatherpurcell.org/calendar,sitemap,,,,[],qwen3:8b,ok,


In [7]:
# Inspect every ministry/category pair and category found in this run.
display(ministries_categories_df)
display(ministry_categories_df)


,name,category,page_count
0,Adult Faith Formation,Faith Formation,1
1,Exceptional Children Support,Pastoral Care,1
2,OCIA (Order of Christian Initiation for Adults),Sacramental Preparation,1
3,Exceptional Children Services,Service and Outreach,1
4,Boy Scouts,Youth and Young Adults,1


,category,ministry_count,page_count
0,Faith Formation,1,1
1,Pastoral Care,1,1
2,Sacramental Preparation,1,1
3,Service and Outreach,1,1
4,Youth and Young Adults,1,1
